# **Bronze audit table**
## creates an empty Delta table

In [6]:
from pyspark.sql import DataFrame
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from pyspark.sql import functions as F

# Audit table structure

AUDIT_TABLE = "bronze_ingestion_audit"

AUDIT_SCHEMA = StructType([
    StructField("batch_id", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("source_path", StringType(), True),
    StructField("load_status", StringType(), False),
    StructField("load_timestamp", TimestampType(), False),
    StructField("error_message", StringType(), True)
])

# Create Bronze audit table

def create_audit_table() -> None:
    """Create the Bronze ingestion audit table if it does not exist."""

    empty_df = spark.createDataFrame([], AUDIT_SCHEMA)

    (empty_df.write
        .format("delta")
        .mode("ignore")
        .saveAsTable(AUDIT_TABLE))

    print(f"Audit table ready: {AUDIT_TABLE}")

StatementMeta(, aacc9c8e-a0ee-4edd-aaf9-8e3d4114b34e, 8, Finished, Available, Finished, False)

### **Execute the function**

In [7]:
create_audit_table()

StatementMeta(, aacc9c8e-a0ee-4edd-aaf9-8e3d4114b34e, 9, Finished, Available, Finished, False)

Audit table ready: bronze_ingestion_audit


### **Verify the table**

In [8]:
# Verify audit table

spark.sql("SHOW TABLES").show()

spark.table(AUDIT_TABLE).printSchema()

display(spark.table(AUDIT_TABLE))

StatementMeta(, aacc9c8e-a0ee-4edd-aaf9-8e3d4114b34e, 10, Finished, Available, Finished, False)

+--------------------+--------------------+-----------+
|           namespace|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|Banking_Fraud_Ana...|bronze_ingestion_...|      false|
+--------------------+--------------------+-----------+

root
 |-- batch_id: string (nullable = true)
 |-- table_name: string (nullable = true)
 |-- source_path: string (nullable = true)
 |-- load_status: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- error_message: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 57725ce2-ae2d-4238-9a91-5194d45d52d8)

### **Audit logging function**

In [9]:
# Audit logging function

def log_ingestion(
    batch_id: str,
    table_name: str,
    source_path: str,
    load_status: str,
    error_message: str = None
) -> None:
    """Record an ingestion result in the Bronze audit table."""

    audit_data = [(
        batch_id,
        table_name,
        source_path,
        load_status,
        None,
        error_message
    )]

    audit_df = spark.createDataFrame(
        audit_data,
        AUDIT_SCHEMA
    )

    audit_df = audit_df.withColumn(
        "load_timestamp",
        F.current_timestamp()
    )

    (audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE))

    print(f"{table_name:<15} {load_status}")

StatementMeta(, aacc9c8e-a0ee-4edd-aaf9-8e3d4114b34e, 11, Finished, Available, Finished, False)

In [10]:
# Verify the latest ingestion batch

audit_df = spark.table(AUDIT_TABLE)

latest_batch_id = (
    audit_df
    .filter(F.col("batch_id") != "TEST_001")
    .orderBy(F.col("load_timestamp").desc())
    .select("batch_id")
    .first()["batch_id"]
)

print(f"Latest batch: {latest_batch_id}")

# Display audit records for this batch

latest_batch_df = (
    audit_df
    .filter(F.col("batch_id") == latest_batch_id)
    .orderBy("table_name")
)

display(latest_batch_df)

# Summarise ingestion results

display(
    latest_batch_df
    .groupBy("load_status")
    .agg(
        F.count("*").alias("total_tables")
    )
)

StatementMeta(, aacc9c8e-a0ee-4edd-aaf9-8e3d4114b34e, 12, Finished, Available, Finished, False)

Latest batch: 20261107_001955


SynapseWidget(Synapse.DataFrame, 55afd1a9-4773-417d-8c19-ffcdb4f3cc70)

SynapseWidget(Synapse.DataFrame, 519119e4-a665-4984-90f1-75eec2e38f43)